In [1]:
import glob
import pandas as pd

csv_files = glob.glob('*.csv')

for filename in csv_files:
    print(f"📄 文件名: {filename}")
    try:
        # 只读取前 2 行 (nrows=1 读取数据行，header默认是第一行)
        # 注意：pd.read_csv 默认把第一行当做表头，nrows=1 会读入表头 + 1行数据
        df = pd.read_csv(filename, nrows=1)
        
        # 打印表头（列名）
        print(f"   第一行 (列名): {list(df.columns)}")
        
        # 打印第一行数据 (如果文件不为空)
        if not df.empty:
            print(f"   第二行 (数值): {df.iloc[0].tolist()}")
            
    except Exception as e:
        print(f"   读取出错: {e}")
    print("-" * 40)

📄 文件名: NACC.csv
   第一行 (列名): ['path', 'filename', 'ID', 'NC', 'MCI', 'DE', 'COG', 'AD', 'PD', 'FTD', 'VD', 'LBD', 'PDD', 'DLB', 'OTHER', 'ADD', 'gender', 'education', 'race', 'hispanic', 'apoe', 'age', 'mmse', 'moca', 'cdr', 'cdrSum', 'boston', 'digitB', 'digitBL', 'digitF', 'digitFL', 'animal', 'Fwords', 'gds', 'lm_imm', 'lm_del', 'lm_memtime', 'craft_imm', 'craft_del', 'mint', 'numberB', 'numberBL', 'numberF', 'numberFL', 'trailA', 'trailB', 'npiq_DEL', 'npiq_HALL', 'npiq_AGIT', 'npiq_DEPD', 'npiq_ANX', 'npiq_ELAT', 'npiq_APA', 'npiq_DISN', 'npiq_IRR', 'npiq_MOT', 'npiq_NITE', 'npiq_APP', 'faq_BILLS', 'faq_TAXES', 'faq_SHOPPING', 'faq_GAMES', 'faq_STOVE', 'faq_MEALPREP', 'faq_EVENTS', 'faq_PAYATTN', 'faq_REMDATES', 'faq_TRAVEL', 'his_PRIMLANG', 'his_HANDED', 'his_LIVSIT', 'his_INDEPEND', 'his_RESIDENC', 'his_MARISTAT', 'his_INRELTO', 'his_NACCFAM', 'his_NACCNREX', 'his_CVHATT', 'his_CVAFIB', 'his_CVANGIO', 'his_CVBYPASS', 'his_CVPACE', 'his_CVPACDEF', 'his_CVCHF', 'his_CVOTHR', 'his_

In [3]:
import pandas as pd
import os
import shutil
import numpy as np
from tqdm import tqdm

# ================= 配置区域 =================

# 1. 原始 CSV 文件路径
source_csv_path = 'NACC.csv'

# 2. 【核心修正】图片所在的文件夹名称 (相对路径)
# 既然图片在当前目录下的 MRI 文件夹里，这里直接写 'MRI'
mri_source_dir = 'MRI' 

# 3. 输出 CSV 的目标列顺序 
target_columns = [
    'path', 'filename', 'ID', 'age', 'gender', 'education', 'hispanic', 'race', 'apoe', 
    'NC', 'MCI', 'DE', 'COG', 'AD', 'mmse', 'moca', 'cdr', 'cdrSum', 'boston', 
    'digitB', 'digitBL', 'digitF', 'digitFL', 'animal', 'Fwords', 'gds', 'lm_imm', 
    'lm_del', 'lm_memtime', 'craft_imm', 'craft_del', 'mint', 'numberB', 'numberBL', 
    'numberF', 'numberFL', 'trailA', 'trailB', 'npiq_DEL', 'npiq_HALL', 'npiq_AGIT', 
    'npiq_DEPD', 'npiq_ANX', 'npiq_ELAT', 'npiq_APA', 'npiq_DISN', 'npiq_IRR', 
    'npiq_MOT', 'npiq_NITE', 'npiq_APP', 'faq_BILLS', 'faq_TAXES', 'faq_SHOPPING', 
    'faq_GAMES', 'faq_STOVE', 'faq_MEALPREP', 'faq_EVENTS', 'faq_PAYATTN', 
    'faq_REMDATES', 'faq_TRAVEL', 'his_PRIMLANG', 'his_HANDED', 'his_LIVSIT', 
    'his_INDEPEND', 'his_RESIDENC', 'his_MARISTAT', 'his_INRELTO', 'his_NACCFAM', 
    'his_NACCNREX', 'his_CVHATT', 'his_CVAFIB', 'his_CVANGIO', 'his_CVBYPASS', 
    'his_CVPACE', 'his_CVPACDEF', 'his_CVCHF', 'his_CVOTHR', 'his_CBSTROKE', 
    'his_CBTIA', 'his_SEIZURES', 'his_TBI', 'his_TBIBRIEF', 'his_TBIEXTEN', 
    'his_TBIWOLOS', 'his_TRAUMBRF', 'his_TRAUMEXT', 'his_TRAUMCHR', 'his_HYPERTEN', 
    'his_HYPERCHO', 'his_DIABETES', 'his_B12DEF', 'his_THYROID', 'his_INCONTU', 
    'his_INCONTF', 'his_DEP2YRS', 'his_DEPOTHR', 'his_PSYCDIS', 'his_NPSYDEV', 
    'his_OCD', 'his_ANXIETY', 'his_SCHIZ', 'his_BIPOLAR', 'his_PTSD', 'his_ALCOHOL', 
    'his_TOBAC100', 'his_SMOKYRS', 'his_PACKSPER', 'his_ABUSOTHR'
]

# ================= 处理逻辑 =================

def process_nacc_split():
    # 检查源图片文件夹是否存在
    if not os.path.exists(mri_source_dir):
        print(f"错误: 当前目录下找不到文件夹 '{mri_source_dir}'。请确认文件夹名称是否正确。")
        return

    print(f"正在读取 {source_csv_path}...")
    try:
        df = pd.read_csv(source_csv_path)
    except FileNotFoundError:
        print(f"错误: 未找到 CSV 文件 {source_csv_path}")
        return

    # 预处理：只提取文件名 (去掉可能存在的路径前缀)
    # 例如 /home/wcj/data/MRI/abc.nii -> abc.nii
    df['filename_clean'] = df['filename'].apply(lambda x: os.path.basename(str(x)))

    # 定义分类映射 (COG: 0->Normal, 1->MCI, 2->AD)
    # 注意：这里直接生成到当前目录下的文件夹，不套 processed_NACC_Split
    categories = {
        0: {'name': 'normal', 'folder': 'NACC_nii_no',  'csv_name': 'NACC_normal.csv'},
        1: {'name': 'mci',    'folder': 'NACC_mci',     'csv_name': 'NACC_mci.csv'},
        2: {'name': 'ad',     'folder': 'NACC_nii_ad',  'csv_name': 'NACC_ad.csv'}
    }

    total_copied = 0
    total_missing = 0

    for cog_val, config in categories.items():
        print(f"\n========== 处理类别: {config['name']} (COG={cog_val}) ==========")
        
        # 筛选数据
        sub_df = df[df['COG'] == cog_val].copy()
        
        # 创建目标文件夹 (例如当前目录下的 NACC_nii_ad)
        target_dir = config['folder']
        if not os.path.exists(target_dir):
            os.makedirs(target_dir)
            
        valid_rows = []
        
        # 遍历每一行
        for idx, row in tqdm(sub_df.iterrows(), total=sub_df.shape[0]):
            pure_filename = row['filename_clean']
            
            # 【关键】直接去当前目录下的 MRI 文件夹里找
            src_path = os.path.join(mri_source_dir, pure_filename)
            dst_path = os.path.join(target_dir, pure_filename)
            
            if os.path.exists(src_path):
                # 复制文件
                shutil.copy2(src_path, dst_path)
                
                # 更新 CSV 数据：
                # 1. path 更新为新文件夹的绝对路径 (确保训练脚本能读到)
                row['path'] = os.path.abspath(target_dir) + os.sep
                # 2. filename 确保是干净的文件名
                row['filename'] = pure_filename
                
                valid_rows.append(row)
                total_copied += 1
            else:
                # 找不到文件，跳过该行，不计入 CSV
                total_missing += 1

        # 生成该类别的 CSV
        if valid_rows:
            processed_df = pd.DataFrame(valid_rows)
            
            # 按照您要求的列顺序重组
            final_df = pd.DataFrame()
            for col in target_columns:
                if col in processed_df.columns:
                    final_df[col] = processed_df[col]
                else:
                    final_df[col] = np.nan # 缺失列填空
            
            # 保存到当前目录
            final_df.to_csv(config['csv_name'], index=False)
            print(f"-> 已生成 CSV: {config['csv_name']} (包含 {len(final_df)} 条数据)")
        else:
            print(f"-> 警告: 类别 {config['name']} 在 '{mri_source_dir}' 中未找到任何文件，未生成 CSV。")

    print("\n================ 总结 ================")
    print(f"处理完成。")
    print(f"成功匹配并复制文件: {total_copied}")
    print(f"缺失文件 (未包含在CSV中): {total_missing}")
    if total_copied == 0:
        print("错误: 依然没有找到任何文件。请确认 'MRI' 文件夹就在脚本旁边，且里面确实有 .nii 文件。")

if __name__ == '__main__':
    process_nacc_split()

正在读取 NACC.csv...

========== 处理类别: normal (COG=0) ==========


100%|██████████| 2299/2299 [02:05<00:00, 18.37it/s] 
/tmp/ipykernel_2872/952836515.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[col] = processed_df[col]
/tmp/ipykernel_2872/952836515.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[col] = processed_df[col]
/tmp/ipykernel_2872/952836515.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1)

-> 已生成 CSV: NACC_normal.csv (包含 2299 条数据)

========== 处理类别: mci (COG=1) ==========


100%|██████████| 944/944 [00:48<00:00, 19.57it/s]
/tmp/ipykernel_2872/952836515.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[col] = processed_df[col]
/tmp/ipykernel_2872/952836515.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[col] = processed_df[col]
/tmp/ipykernel_2872/952836515.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) in

-> 已生成 CSV: NACC_mci.csv (包含 944 条数据)

========== 处理类别: ad (COG=2) ==========


100%|██████████| 779/779 [00:36<00:00, 21.07it/s]


-> 已生成 CSV: NACC_ad.csv (包含 779 条数据)

================ 总结 ================
处理完成。
成功匹配并复制文件: 4022
缺失文件 (未包含在CSV中): 0


/tmp/ipykernel_2872/952836515.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[col] = processed_df[col]
/tmp/ipykernel_2872/952836515.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[col] = processed_df[col]
/tmp/ipykernel_2872/952836515.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe

In [4]:
import os

# 定义你的三个数据目录
folders = ['NACC_nii_ad', 'NACC_nii_no', 'NACC_mci']

def clean_empty_files(folder_list):
    count = 0
    for folder in folder_list:
        if not os.path.exists(folder):
            print(f"目录不存在: {folder}")
            continue
            
        print(f"正在检查目录: {folder} ...")
        for filename in os.listdir(folder):
            file_path = os.path.join(folder, filename)
            
            # 检查是否是文件且以 .nii 结尾
            if os.path.isfile(file_path) and filename.endswith('.nii'):
                # 检查文件大小是否为 0
                if os.path.getsize(file_path) == 0:
                    try:
                        os.remove(file_path)
                        print(f"已删除空文件: {file_path}")
                        count += 1
                    except OSError as e:
                        print(f"删除失败: {file_path}, 错误: {e}")
    
    print(f"--------------------------")
    print(f"检查完成，共删除了 {count} 个空文件。")

# 执行清理
clean_empty_files(folders)

正在检查目录: NACC_nii_ad ...
已删除空文件: NACC_nii_ad/NACC497997_128401136192134176253428335081351187731650.nii
已删除空文件: NACC_nii_ad/NACC027205_12840113619641050359208386302499410896159485470824307.nii
已删除空文件: NACC_nii_ad/NACC720436_12840113619640845981644853101483708767722843770249972.nii
已删除空文件: NACC_nii_ad/NACC916290_128401136192134176253428338181342538481920.nii
已删除空文件: NACC_nii_ad/mri7069.nii
已删除空文件: NACC_nii_ad/NACC683064_128401136192134176253428326311215104070689.nii
已删除空文件: NACC_nii_ad/NACC205154_13122110752436707830000020012111453209400000031.nii
已删除空文件: NACC_nii_ad/NACC271933_128401136192134176253428319601354034346802.nii
已删除空文件: NACC_nii_ad/NACC269863_128401136192134176253428321521212526881795.nii
已删除空文件: NACC_nii_ad/NACC426385_128401136192134176253428319391304963023663.nii
已删除空文件: NACC_nii_ad/mri132.nii
已删除空文件: NACC_nii_ad/NACC774848_134667058911171845030202012050209431271000.nii
已删除空文件: NACC_nii_ad/NACC265729_128401136192134176253428329101273791482586.nii
已删除空文件: NACC_nii_ad/NACC2130

In [5]:
import pandas as pd
import os

# 配置：CSV 文件名与其对应的图片存储文件夹
# 请确保这些文件夹和 CSV 都在当前运行目录下
tasks = [
    {'csv': 'NACC_normal.csv', 'folder': 'NACC_nii_no'},
    {'csv': 'NACC_mci.csv',    'folder': 'NACC_mci'},
    {'csv': 'NACC_ad.csv',     'folder': 'NACC_nii_ad'}
]

def sync_csv_with_files():
    print("🚀 开始同步 CSV 与物理文件...")
    
    total_removed = 0
    
    for task in tasks:
        csv_path = task['csv']
        folder_path = task['folder']
        
        if not os.path.exists(csv_path):
            print(f"⚠️ 跳过: 找不到 CSV 文件 {csv_path}")
            continue
            
        if not os.path.exists(folder_path):
            print(f"⚠️ 跳过: 找不到文件夹 {folder_path}")
            continue

        # 读取 CSV
        df = pd.read_csv(csv_path)
        original_count = len(df)
        
        # 核心逻辑：判断文件是否存在
        # 使用 apply 方法遍历每一行，检查 folder/filename 是否存在
        # axis=1 表示按行操作
        def check_file_exists(row):
            # 组合文件的相对路径
            file_path = os.path.join(folder_path, str(row['filename']))
            return os.path.exists(file_path)

        # 获取所有存在的文件的布尔索引
        mask = df.apply(check_file_exists, axis=1)
        
        # 过滤 DataFrame
        df_cleaned = df[mask]
        
        # 计算删除了多少行
        removed_count = original_count - len(df_cleaned)
        total_removed += removed_count
        
        if removed_count > 0:
            # 覆盖保存 CSV
            df_cleaned.to_csv(csv_path, index=False)
            print(f"✅ {csv_path}: \t原数据 {original_count} 条 -> \t清洗后 {len(df_cleaned)} 条 \t(删除了 {removed_count} 条无效索引)")
        else:
            print(f"🆗 {csv_path}: \t数据完整，无需更改。")

    print("-" * 50)
    print(f"🎉 全部完成！共删除了 {total_removed} 条指向不存在文件的记录。")

if __name__ == '__main__':
    sync_csv_with_files()

🚀 开始同步 CSV 与物理文件...
✅ NACC_normal.csv: 	原数据 2299 条 -> 	清洗后 2116 条 	(删除了 183 条无效索引)
✅ NACC_mci.csv: 	原数据 944 条 -> 	清洗后 879 条 	(删除了 65 条无效索引)
✅ NACC_ad.csv: 	原数据 779 条 -> 	清洗后 713 条 	(删除了 66 条无效索引)
--------------------------------------------------
🎉 全部完成！共删除了 314 条指向不存在文件的记录。
